# Modul 4: Nichtlineare Systeme

---

## Der Übergang zur vollen Komplexität

Lineare Systeme sind lehrreich, aber die Welt ist nichtlinear. In diesem Modul:

1. **Linearisierung**: Wie lineare Analyse trotzdem hilft (lokal)
2. **Grenzzyklen**: Ein fundamental neues Phänomen
3. **Poincaré-Bendixson**: Warum auch in 2D kein Chaos möglich ist

Der rote Faden: Selbst bei nichtlinearen Systemen gibt es **Struktursätze**, die das mögliche Verhalten einschränken.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from scipy.linalg import eig

plt.rcParams.update({
    'figure.figsize': (12, 10),
    'figure.dpi': 100,
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'lines.linewidth': 1.5,
})

%matplotlib inline

---

## 4.1 Linearisierung: Die Brücke vom Nichtlinearen zum Linearen

### Die zentrale Idee

Gegeben ein nichtlineares System:
$$\dot{\mathbf{x}} = \mathbf{f}(\mathbf{x})$$

Mit Fixpunkt $\mathbf{x}^*$ (also $\mathbf{f}(\mathbf{x}^*) = 0$).

**Taylor-Entwicklung** um $\mathbf{x}^*$:
$$\mathbf{f}(\mathbf{x}^* + \boldsymbol{\epsilon}) \approx \underbrace{\mathbf{f}(\mathbf{x}^*)}_{=0} + D\mathbf{f}(\mathbf{x}^*) \cdot \boldsymbol{\epsilon} + O(\epsilon^2)$$

Die **Jacobi-Matrix** $D\mathbf{f}$ ist:
$$D\mathbf{f} = \begin{pmatrix} \frac{\partial f_1}{\partial x_1} & \frac{\partial f_1}{\partial x_2} \\ \frac{\partial f_2}{\partial x_1} & \frac{\partial f_2}{\partial x_2} \end{pmatrix}$$

Für kleine Störungen $\boldsymbol{\epsilon}$ gilt also:
$$\dot{\boldsymbol{\epsilon}} \approx J \cdot \boldsymbol{\epsilon}$$

wobei $J = D\mathbf{f}(\mathbf{x}^*)$. Das ist ein lineares System!

In [ ]:
def compute_jacobian(f, x, h=1e-8):
    """
    Berechnet die Jacobi-Matrix numerisch.
    
    f: Funktion, die [x, y] -> [dx/dt, dy/dt] abbildet
    x: Punkt, an dem die Jacobi-Matrix berechnet wird
    """
    n = len(x)
    J = np.zeros((n, n))
    
    f0 = np.array(f(x))
    
    for i in range(n):
        x_plus = x.copy()
        x_plus[i] += h
        x_minus = x.copy()
        x_minus[i] -= h
        
        J[:, i] = (np.array(f(x_plus)) - np.array(f(x_minus))) / (2 * h)
    
    return J


def analyze_fixed_point(f, fp, name=""):
    """
    Analysiert einen Fixpunkt durch Linearisierung.
    """
    J = compute_jacobian(f, fp)
    eigenvalues = np.linalg.eigvals(J)
    
    print(f"\n{'='*50}")
    print(f"Fixpunkt: {name} = {fp}")
    print(f"{'='*50}")
    print(f"\nJacobi-Matrix J:")
    print(f"  [{J[0,0]:8.4f}  {J[0,1]:8.4f}]")
    print(f"  [{J[1,0]:8.4f}  {J[1,1]:8.4f}]")
    print(f"\nEigenwerte: {eigenvalues}")
    print(f"Spur τ = {np.trace(J):.4f}")
    print(f"Determinante Δ = {np.linalg.det(J):.4f}")
    
    # Klassifikation
    real_parts = np.real(eigenvalues)
    imag_parts = np.imag(eigenvalues)
    
    if np.any(np.abs(imag_parts) > 1e-10):
        if real_parts[0] < -1e-10:
            classification = "Stabile Spirale"
        elif real_parts[0] > 1e-10:
            classification = "Instabile Spirale"
        else:
            classification = "Zentrum (nichtlinear: weitere Analyse nötig!)"
    else:
        ev_real = np.real(eigenvalues)
        if ev_real[0] * ev_real[1] < 0:
            classification = "Sattelpunkt"
        elif np.all(ev_real < 0):
            classification = "Stabiler Knoten"
        elif np.all(ev_real > 0):
            classification = "Instabiler Knoten"
        else:
            classification = "Degeneriert"
    
    print(f"\n→ Klassifikation: {classification}")
    
    return J, eigenvalues, classification

In [ ]:
# Beispiel: Das gedämpfte Pendel
# θ'' + bθ' + sin(θ) = 0
# Als System: x = θ, y = θ'
# ẋ = y
# ẏ = -sin(x) - b*y

b = 0.5  # Dämpfung

def pendulum(state):
    x, y = state
    return [y, -np.sin(x) - b * y]

# Fixpunkte: sin(x) = 0 und y = 0
# Also: (0, 0), (π, 0), (2π, 0), ...

analyze_fixed_point(pendulum, np.array([0.0, 0.0]), "Ruhelage unten")
analyze_fixed_point(pendulum, np.array([np.pi, 0.0]), "Ruhelage oben")

### Der Satz von Hartman-Grobman

**Informelle Version:**

> Wenn alle Eigenwerte der Jacobi-Matrix $J$ Realteil $\neq 0$ haben, dann ist das nichtlineare System **lokal topologisch äquivalent** zum linearisierten System.

**Was bedeutet "topologisch äquivalent"?**

Die Phasenportraits können durch eine stetige Deformation (Homöomorphismus) ineinander überführt werden. Die qualitative Struktur ist identisch:
- Spiralen bleiben Spiralen
- Knoten bleiben Knoten
- Sättel bleiben Sättel

**Wann versagt Hartman-Grobman?**

Wenn ein Eigenwert Realteil 0 hat (auf der imaginären Achse). Dann entscheiden die nichtlinearen Terme! Ein linearisiertes Zentrum könnte nichtlinear eine Spirale sein.

In [ ]:
# Visualisierung: Nichtlineares Pendel vs. Linearisierung

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Gemeinsame Funktion für Phasenportrait
def plot_phase_portrait(system, ax, title, x_range=(-4, 4), y_range=(-3, 3)):
    # Vektorfeld
    x = np.linspace(x_range[0], x_range[1], 20)
    y = np.linspace(y_range[0], y_range[1], 20)
    X, Y = np.meshgrid(x, y)
    
    U = np.zeros_like(X)
    V = np.zeros_like(Y)
    
    for i in range(len(x)):
        for j in range(len(y)):
            dxdt = system([X[j, i], Y[j, i]])
            U[j, i] = dxdt[0]
            V[j, i] = dxdt[1]
    
    M = np.sqrt(U**2 + V**2)
    M[M == 0] = 1
    ax.quiver(X, Y, U/M, V/M, M, cmap='coolwarm', alpha=0.5)
    
    # Trajektorien
    t = np.linspace(0, 20, 1000)
    
    for x0 in np.linspace(-3, 3, 7):
        for y0 in [-2, 0, 2]:
            sol = odeint(lambda s, t: system(s), [x0, y0], t)
            ax.plot(sol[:, 0], sol[:, 1], 'b-', alpha=0.5, linewidth=0.8)
    
    ax.set_xlabel(r'$\theta$')
    ax.set_ylabel(r'$\dot{\theta}$')
    ax.set_title(title)
    ax.set_xlim(x_range)
    ax.set_ylim(y_range)

# Nichtlineares Pendel
plot_phase_portrait(pendulum, axes[0], f'Nichtlineares Pendel (b={b})')
axes[0].plot(0, 0, 'go', markersize=12, label='Stabile Spirale')
axes[0].plot(np.pi, 0, 'ro', markersize=12, fillstyle='none', markeredgewidth=2, label='Sattel')
axes[0].plot(-np.pi, 0, 'ro', markersize=12, fillstyle='none', markeredgewidth=2)
axes[0].legend()

# Linearisiertes System um (0,0): ẋ = y, ẏ = -x - by
def linearized_pendulum(state):
    x, y = state
    return [y, -x - b * y]

plot_phase_portrait(linearized_pendulum, axes[1], 'Linearisiert um (0,0)')
axes[1].plot(0, 0, 'go', markersize=12)

plt.tight_layout()
plt.show()

### Beobachtung

**Lokal** (nahe dem Ursprung) sehen beide Systeme identisch aus - stabile Spirale.

**Global** zeigt das nichtlineare System reichere Struktur:
- Periodische "Durchdrehungen" (wenn genug Energie)
- Sattelpunkte bei $\theta = \pm\pi$ (Pendel oben)
- Separatrizen, die verschiedene Bewegungstypen trennen

Die Linearisierung ist ein **lokales** Werkzeug. Für globale Aussagen brauchen wir andere Methoden.

---

## 4.2 Grenzzyklen: Ein fundamental neues Phänomen

Ein **Grenzzyklus** ist eine **isolierte** geschlossene Trajektorie.

- "Isoliert" = Nicht Teil einer Familie von Kreisen (wie beim Zentrum)
- Benachbarte Trajektorien spiralen darauf zu oder davon weg

**Grenzzyklen existieren nur in nichtlinearen Systemen!**

Sie repräsentieren **selbsterhaltende Oszillationen** - das System schwingt, ohne dass äußere Anregung nötig ist.

In [ ]:
# Der Van-der-Pol Oszillator: Das Paradebeispiel
# ẍ - μ(1 - x²)ẋ + x = 0
# Als System: ẋ = y, ẏ = μ(1-x²)y - x

def van_der_pol(state, t, mu=1.0):
    x, y = state
    return [y, mu * (1 - x**2) * y - x]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

mu_values = [0.1, 1.0, 5.0]

for ax, mu in zip(axes, mu_values):
    # Trajektorien von verschiedenen Startpunkten
    t = np.linspace(0, 50, 2000)
    
    # Von innen (nahe Ursprung)
    for x0 in [0.1, 0.5]:
        sol = odeint(van_der_pol, [x0, 0], t, args=(mu,))
        ax.plot(sol[:, 0], sol[:, 1], 'b-', alpha=0.7, linewidth=1)
    
    # Von außen
    for x0 in [4, 5]:
        sol = odeint(van_der_pol, [x0, 0], t, args=(mu,))
        ax.plot(sol[:, 0], sol[:, 1], 'r-', alpha=0.7, linewidth=1)
    
    # Der Grenzzyklus selbst (lange integrieren, Transient abschneiden)
    t_long = np.linspace(0, 200, 10000)
    sol_long = odeint(van_der_pol, [0.1, 0], t_long, args=(mu,))
    ax.plot(sol_long[-2000:, 0], sol_long[-2000:, 1], 'k-', linewidth=2.5,
           label='Grenzzyklus')
    
    ax.plot(0, 0, 'ko', markersize=8, fillstyle='none', markeredgewidth=2)
    ax.set_xlabel(r'$x$')
    ax.set_ylabel(r'$\dot{x}$')
    ax.set_title(f'Van-der-Pol Oszillator (μ = {mu})')
    ax.set_aspect('equal')
    ax.legend()
    
plt.tight_layout()
plt.show()

### Die Physik des Van-der-Pol Oszillators

Der Term $\mu(1 - x^2)\dot{x}$ ist eine **nichtlineare Dämpfung**:

- Für $|x| < 1$: **Negative Dämpfung** → Energie wird *hinzugefügt*
- Für $|x| > 1$: **Positive Dämpfung** → Energie wird *entzogen*

Das System balanciert sich selbst aus:
- Kleine Schwingungen werden verstärkt
- Große Schwingungen werden gedämpft
- Ergebnis: Stabile Oszillation mit fester Amplitude

Dies modelliert:
- Herz-Rhythmus
- Elektronische Oszillatoren
- Populationszyklen
- Wirtschaftszyklen

Der Parameter $\mu$ steuert die **Relaxation**:
- Klein: Fast sinusförmig
- Groß: Abrupte "Relaxationsoszillationen" (schnelle Sprünge, langsames Kriechen)

In [ ]:
# Zeitverlauf zeigt die Relaxationsoszillation

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

t = np.linspace(0, 50, 2000)

# Kleines μ: fast sinusförmig
mu = 0.1
sol = odeint(van_der_pol, [0.1, 0], t, args=(mu,))
axes[0].plot(t, sol[:, 0], 'b-', linewidth=1.5)
axes[0].set_xlabel(r'$t$')
axes[0].set_ylabel(r'$x(t)$')
axes[0].set_title(f'μ = {mu}: Fast harmonisch')

# Großes μ: Relaxationsoszillation
mu = 5.0
sol = odeint(van_der_pol, [0.1, 0], t, args=(mu,))
axes[1].plot(t, sol[:, 0], 'b-', linewidth=1.5)
axes[1].set_xlabel(r'$t$')
axes[1].set_ylabel(r'$x(t)$')
axes[1].set_title(f'μ = {mu}: Relaxationsoszillation')

plt.tight_layout()
plt.show()

---

## 4.3 Der Satz von Poincaré-Bendixson

Einer der tiefsten Sätze der 2D-Dynamik:

**Satz (Poincaré-Bendixson):**

> Sei $R$ eine beschränkte, abgeschlossene Region in der Ebene, die **keinen Fixpunkt** enthält. Wenn eine Trajektorie für alle $t > 0$ in $R$ bleibt, dann ist sie entweder:
> 1. Selbst ein geschlossener Orbit (Grenzzyklus), oder
> 2. Sie strebt gegen einen geschlossenen Orbit.

### Konsequenz: Kein Chaos in 2D!

Das Langzeitverhalten in 2D ist beschränkt auf:
- Fixpunkte
- Periodische Orbits (Grenzzyklen)
- Verbindungen zwischen Fixpunkten (hetero-/homokline Orbits)

**Kein aperiodisches, beschränktes, sensitives Verhalten möglich!**

Für Chaos brauchen wir **mindestens 3 Dimensionen** (kontinuierliche Systeme) oder **1 Dimension** (diskrete Abbildungen).

### Warum funktioniert das?

In 2D können Trajektorien sich nicht schneiden (Eindeutigkeit). Eine Trajektorie, die in einer beschränkten Region bleibt, muss irgendwo hin:

- Zu einem Fixpunkt? → Wir haben angenommen, es gibt keinen in $R$
- Gegen unendlich? → $R$ ist beschränkt
- Also: Sie muss sich "einwickeln" → geschlossener Orbit

In 3D fällt dieses Argument weg: Trajektorien können "aneinander vorbeiziehen", ohne sich zu schneiden. Das ermöglicht Chaos.

---

## 4.4 Kriterien für die Existenz von Grenzzyklen

### Bendixsons Kriterium (negativ)

**Satz:** Wenn $\nabla \cdot \mathbf{f} = \frac{\partial f}{\partial x} + \frac{\partial g}{\partial y}$ in einer **einfach zusammenhängenden** Region **einheitliches Vorzeichen** hat, dann gibt es dort **keinen geschlossenen Orbit**.

**Intuition:** $\nabla \cdot \mathbf{f}$ ist die Divergenz - sie misst, ob der Fluss Volumen (in 2D: Fläche) komprimiert oder expandiert. Ein geschlossener Orbit hätte beides im Gleichgewicht.

In [ ]:
# Beispiel: Anwendung von Bendixsons Kriterium

# System: ẋ = x(2 - x - y), ẏ = y(4x - x² - 3)
# Divergenz: ∂f/∂x + ∂g/∂y = (2 - 2x - y) + (4x - x² - 3)
#                          = -1 - 2x - y + 4x - x²
#                          = -1 + 2x - x² - y
#                          = -(x² - 2x + 1) - y
#                          = -(x-1)² - y

def compute_divergence():
    print("System: ẋ = x(2 - x - y), ẏ = y(4x - x² - 3)")
    print("\nDivergenz: ∂f/∂x + ∂g/∂y")
    print("  = (2 - 2x - y) + (4x - x² - 3)")
    print("  = -1 + 2x - x² - y")
    print("  = -(x-1)² - y")
    print("\nFür y > 0 (obere Halbebene):")
    print("  -(x-1)² ≤ 0 und -y < 0")
    print("  → Divergenz < 0 überall in y > 0")
    print("\n→ Nach Bendixson: Kein geschlossener Orbit in y > 0!")

compute_divergence()

### Liénard-Systeme: Eine wichtige Klasse

Systeme der Form:
$$\ddot{x} + f(x)\dot{x} + g(x) = 0$$

haben unter bestimmten Bedingungen **genau einen** stabilen Grenzzyklus.

**Satz (Liénard):** Wenn
- $f$ und $g$ stetig
- $g$ ungerade mit $g(x) > 0$ für $x > 0$
- $F(x) = \int_0^x f(s) ds$ ist ungerade
- $F(x) < 0$ für $0 < x < a$, $F(a) = 0$, $F(x) > 0$ und monoton steigend für $x > a$
- $F(x) \to \infty$ für $x \to \infty$

dann hat das System **genau einen Grenzzyklus** (stabil).

Der Van-der-Pol Oszillator erfüllt diese Bedingungen!

---

## 4.5 Globale Analyse: Separatrizen und Einzugsgebiete

Nichtlineare Systeme haben **globale Struktur**, die über lokale Linearisierung hinausgeht.

In [ ]:
# Das gedämpfte Pendel: Globale Struktur

def plot_pendulum_global():
    fig, ax = plt.subplots(figsize=(14, 8))
    
    b = 0.3
    
    def pendulum(state, t):
        x, y = state
        return [y, -np.sin(x) - b * y]
    
    # Viele Trajektorien
    t = np.linspace(0, 30, 1000)
    
    # Verschiedene Energieniveaus
    for x0 in np.linspace(-2*np.pi, 2*np.pi, 25):
        for y0 in np.linspace(-3, 3, 15):
            sol = odeint(pendulum, [x0, y0], t)
            ax.plot(sol[:, 0], sol[:, 1], 'b-', alpha=0.3, linewidth=0.5)
    
    # Separatrizen (vom Sattelpunkt bei π)
    # Diese starten/enden nahe am Sattelpunkt entlang der Eigenrichtungen
    epsilon = 0.01
    
    # Instabile Mannigfaltigkeit (vom Sattel weg)
    for sign in [1, -1]:
        sol = odeint(pendulum, [np.pi + sign*epsilon, 0], t)
        ax.plot(sol[:, 0], sol[:, 1], 'r-', linewidth=2.5, alpha=0.8)
        sol = odeint(pendulum, [-np.pi + sign*epsilon, 0], t)
        ax.plot(sol[:, 0], sol[:, 1], 'r-', linewidth=2.5, alpha=0.8)
    
    # Fixpunkte
    for x_fp in [0, np.pi, -np.pi, 2*np.pi, -2*np.pi]:
        if abs(x_fp) == np.pi or abs(x_fp) == 0:
            if x_fp == 0:
                ax.plot(x_fp, 0, 'go', markersize=12)  # Stabil
            else:
                ax.plot(x_fp, 0, 'ko', markersize=10, fillstyle='none', 
                       markeredgewidth=2)  # Sattel
    
    ax.set_xlabel(r'$\theta$ (Winkel)', fontsize=14)
    ax.set_ylabel(r'$\dot{\theta}$ (Winkelgeschwindigkeit)', fontsize=14)
    ax.set_title(f'Gedämpftes Pendel (b = {b}): Globale Struktur', fontsize=16)
    ax.set_xlim(-2.5*np.pi, 2.5*np.pi)
    ax.set_ylim(-4, 4)
    
    # Erklärungen
    ax.annotate('Stabiles Gleichgewicht\n(Pendel unten)', 
               xy=(0, 0), xytext=(0.5, -2.5), fontsize=10,
               arrowprops=dict(arrowstyle='->', color='green'))
    ax.annotate('Sattelpunkt\n(Pendel oben)', 
               xy=(np.pi, 0), xytext=(np.pi+0.5, 1.5), fontsize=10,
               arrowprops=dict(arrowstyle='->', color='black'))
    ax.annotate('Separatrix', xy=(np.pi+0.5, 0.5), xytext=(np.pi+1, 2.5),
               fontsize=10, color='red',
               arrowprops=dict(arrowstyle='->', color='red'))
    
    return fig

plot_pendulum_global()
plt.show()

### Die Separatrix als Grenze

Die **Separatrix** (rot) trennt verschiedene **Einzugsgebiete** (basins of attraction):

- **Innerhalb**: Trajektorien spiralen zum nächsten stabilen Gleichgewicht
- **Außerhalb**: Trajektorien machen eine oder mehrere Umdrehungen, bevor sie sich beruhigen

Die Separatrix verbindet Sattelpunkte - sie ist Teil der stabilen/instabilen Mannigfaltigkeit dieser Punkte.

Diese globale Struktur ist **topologisch robust**: Kleine Parameteränderungen verzerren sie, zerstören sie aber nicht.

---

## 4.6 Zusammenfassung

### Kernkonzepte

1. **Linearisierung**: Jacobi-Matrix an Fixpunkten gibt lokales Verhalten. Hartman-Grobman sagt: Lokal ist das Verhalten wie linear (wenn keine Eigenwerte auf der imaginären Achse).

2. **Grenzzyklen**: Isolierte geschlossene Orbits. Existieren nur nichtlinear. Repräsentieren selbsterhaltende Oszillationen.

3. **Poincaré-Bendixson**: In 2D endet alles bei Fixpunkten oder periodischen Orbits. Kein Chaos möglich.

4. **Bendixsons Kriterium**: Einheitliches Vorzeichen der Divergenz schließt geschlossene Orbits aus.

5. **Globale Struktur**: Separatrizen und Einzugsgebiete strukturieren den Phasenraum.

### Was kommt

Im nächsten Modul: **Bifurkationen** - Was passiert, wenn Parameter verändert werden und sich die qualitative Struktur ändert? Geburt und Tod von Fixpunkten und Grenzzyklen.

---

## Übungen

### Übung 4.1
Analysiere das System $\dot{x} = y$, $\dot{y} = x - x^3 - 0.1y$ (Duffing-Oszillator, gedämpft). Finde alle Fixpunkte und klassifiziere sie.

### Übung 4.2
Zeige mit Bendixsons Kriterium, dass das System $\dot{x} = -x + y + xy$, $\dot{y} = -2y$ keinen Grenzzyklus hat.

### Übung 4.3 (Konzeptuell)
Warum kann ein konservatives System (z.B. Pendel ohne Dämpfung) keinen Grenzzyklus haben? Tipp: Denke an Energieerhaltung.

In [ ]:
# Raum für deine Lösungen

